# Exploratory Pitch Stance Pipeline

This notebook explores a frame-by-frame pipeline for pitch-by-pitch catcher stance analytics.

Goals:
- identify the set-stance window inside each pitch clip
- improve catcher selection and reject batter/umpire swaps
- classify stance per frame and aggregate to a pitch label
- filter bad broadcast angles and normalize coordinates

The production code already contains a strong catcher selector in `src/catcher_detection/detector.py`. This notebook builds on that logic rather than replacing it.

In [ ]:
from pathlib import Path

import pandas as pd

from research.scripts.pitch_stance_research import (
    BASEBALLCV_AVAILABLE,
    analyze_directory,
    event_anchor_window,
    majority_vote,
    rolling_majority_vote,
)

video_dir = Path('data/examples/duke-2026-04-21-liberty-sample/downloads')
summaries = analyze_directory(video_dir)
df = pd.json_normalize(summaries)

BASEBALLCV_AVAILABLE

## BaseballCV / RF-DETR Option Space

Official BaseballCV documentation shows the following baseball-specific aliases that matter for this problem:

- `ball_tracking.pt`: ball trajectory and release/contact anchoring
- `glove_tracking.pt`: glove, ball, home plate, and pitcher rubber tracking
- `pitcher_hitter_catcher.pt`: coarse broadcast triage for the three main people in frame
- `rfdetr_glove_tracking`: RF-DETR version of glove / ball / plate / rubber tracking

That maps cleanly to the three research roles we need:
1. anchor the critical pitch event
2. reject person swaps using spatial context
3. optionally tighten broadcast detection before pose extraction

Primary sources:
- BaseballCV GitHub: https://github.com/BaseballCV/BaseballCV
- RF-DETR GitHub: https://github.com/roboflow/rf-detr


In [ ]:
from research.scripts.pitch_stance_research import option_matrix

option_df = pd.DataFrame(option_matrix())
option_df


## Repository Audit Findings

Relevant existing pieces:
- `src/catcher_detection/detector.py`: plate-area ROI gating, catcher-vs-batter/umpire rejection, score-based abstention
- `src/curator/features.py`: YOLO pose streaming, catcher normalization, fixed-length keypoint export
- `src/stance_pipeline/overlay.py`: frame-by-frame pose rendering during replay
- `src/stance_pipeline/model.py`: MLP stance classifier for pitch-level inference

The current pipeline already solves candidate selection fairly well. The main research gap is temporal: which frames inside a clip should actually contribute to the final pitch stance label?

## Working Hypothesis

The cleanest solution is a hybrid pipeline:

- use BaseballCV PHC proposals to establish catcher identity and the usable broadcast segment
- use the longest plausible BaseballCV ball trajectory as the preferred impact anchor
- search `1.75-0.45s` before impact for a contiguous low-motion set stance
- preserve full-frame `512px` pose inference for compatibility with the trained MLP
- aggregate overlapping seven-frame predictions with pose-quality-weighted voting

This avoids overfitting the whole clip to the pre-pitch stance while still giving a deterministic fallback when BaseballCV or RF-DETR assets are unavailable.

## Objective A: Set Stance Window Identification

The original exploratory summary compacted accepted detections and therefore allowed windows to cross missing frames and camera changes. The corrected harness preserves source frame indices. The implemented event-anchored benchmark produced these results:

| Clip | Verified label | Prediction | Impact error | Set window |
| --- | --- | --- | ---: | --- |
| `...356-369.mp4` | LKD | LKD | 70ms | 586-622 |
| `...378-388.mp4` | LKD | LKD | 66ms | 409-437 |
| `...395-409.mp4` | RKD | RKD | 69ms | 586-622 |
| `...428-444.mp4` | LKD | LKD | 69ms | 623-659 |
| `...449-462.mp4` | LKD | LKD | 86ms | 528-576 |

Interpretation:
- all five ball-trajectory anchors were within 86ms of visual verification
- all five anatomical stance labels matched human verification
- the previous last-seven-valid-frames path flipped four pre-impact LKD clips to post-pitch RKD
- crop pose inference caused feature-domain shift, so PHC guides full-frame pose selection until retraining

The accuracy-first benchmark took 173.6 seconds for five clips on Apple MPS. Generated details are written by `research/scripts/benchmark_staged_pipeline.py`.

In [ ]:
# Reproduce the sample summary table when running interactively.
cols = [
    'video',
    'duration_s',
    'valid_detections',
    'first_valid_frame',
    'last_valid_frame',
    'best_window',
]
df[cols]


## Objective B: Catcher Detection and Disambiguation

The current detector is already doing meaningful work:

- plate-area ROI gating keeps the catcher near the expected home-plate zone
- invalid zones reject dugout and edge-of-frame clutter
- lower-body geometry rejects upright pitcher-like or batter-like candidates
- anchor-distance scoring prefers a catcher-shaped lower-center pose over a generic crouch

On the sample Duke at Liberty clips, the dominant rejection reasons were pitcher-like low/tall geometry, invalid-zone overlap, narrow stance, and hips too far from the catcher anchor. That is the right failure mode for this task because it prevents swapping the catcher with nearby players.

## Objectives B-D: Detection, Classification, and Camera Quality

Objective B, catcher disambiguation:
- use the current plate-anchor ROI and invalid-zone filters as the first line of defense
- keep rejecting tall/low pitcher-like boxes, narrow stances, and near-edge dugout clutter
- add an external detector such as BaseballCV `pitcher_hitter_catcher.pt` only as an optional candidate generator

Objective C, stance classification:
- classify each valid frame with geometric features or a lightweight model
- aggregate over the selected window with majority vote or a rolling median on per-frame labels
- return one pitch-level stance plus confidence and quality metadata

Objective D, camera filtering and normalization:
- reject clips with too many failed frames or unstable plate-anchor geometry
- normalize keypoints by torso or box scale before any per-frame classifier
- use camera-quality flags so bad broadcast angles never silently become stance labels

## Prompt Framing for the Full Problem

The full system should answer one question per pitch clip: what was the catcher doing in the stationary set-stance before the pitch?

That means the implementation should optimize for three things in order:

1. correct temporal window selection
2. correct catcher identity inside the window
3. stable pitch-level stance aggregation

If any step is uncertain, the pipeline should prefer abstaining or marking the clip low-confidence rather than forcing a stance label from post-pitch motion.

In [ ]:
# Example temporal aggregation helpers.
labels = ['Squat', 'Squat', None, 'RKD', 'RKD', 'RKD']
majority_vote(labels), rolling_majority_vote(labels, window=3), event_anchor_window(anchor_frame_idx=240, fps=60.0, pre_seconds=1.5)


## End-to-End Production Shape

The production path should look like this:

1. detect camera cuts while preserving source frame numbers
2. establish a contiguous catcher track with BaseballCV PHC
3. anchor impact with a plausible BaseballCV ball trajectory
4. select a contiguous, stationary pre-impact set window
5. run PHC-guided full-frame pose inference using the training-time feature contract
6. aggregate rolling MLP predictions and abstain when quality is insufficient
7. emit stance, confidence, timing, provenance, votes, and quality flags

The spec for this flow is captured in `research/specs/pitch_stance_pipeline_spec.md`.